# Spectral Synthesis — documentation examples

Companion notebook to the **Spectral Synthesis** documentation page
(`docs/source/spectral_synthesis.rst`). One section per code snippet.

Requires only the core `stellar-spice` install.

In [1]:
%matplotlib inline
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

## Basic usage

In [2]:
from spice.models import IcosphereModel
from spice.spectrum import simulate_observed_flux, Blackbody

bb = Blackbody()
star = IcosphereModel.construct(1000, 1., 1., bb.solar_parameters, bb.parameter_names)

wavelengths = np.linspace(4000., 7000., 2000)   # Angstroms
flux = simulate_observed_flux(bb.intensity, star, np.log10(wavelengths))

print("shape:", flux.shape, "| peak flux:", float(flux[:, 0].max()), "erg/s/cm^2/A at 10 pc")

[spice] IcosphereModel constructed in 0.9 s
shape: (2000, 2) | peak flux: 4.2113687537259636e-11 erg/s/cm^2/A at 10 pc


## Doppler broadening from rotation

An approaching limb contributes blueshifted light, a receding limb redshifted
light — a rotating star's lines broaden. The blackbody continuum is smooth,
so to *see* the effect we use a Gaussian line-profile emulator.

In [3]:
from spice.models.mesh_transform import add_rotation, evaluate_rotation
from spice.spectrum import GaussianLineEmulator

line_center = 5500.0
gle = GaussianLineEmulator(line_centers=[line_center], line_widths=[0.5], line_depths=[0.5])

line_wavelengths = np.linspace(line_center - 5, line_center + 5, 1000)
log_wl = np.log10(line_wavelengths)

static_flux = simulate_observed_flux(gle.intensity, star, log_wl)

fast = evaluate_rotation(add_rotation(star, rotation_velocity=50.), 0.)
rot_flux = simulate_observed_flux(gle.intensity, fast, log_wl)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(line_wavelengths, static_flux[:, 0] / static_flux[:, 1], label="No rotation")
ax.plot(line_wavelengths, rot_flux[:, 0] / rot_flux[:, 1], label="50 km/s")
ax.set_xlabel(r"Wavelength [$\AA$]")
ax.set_ylabel("Normalized flux")
ax.legend()
plt.show()

PHOEBE: passband "Bolometric:900-40000" has a newer version available.  Run phoebe.list_passband_online_history("Bolometric:900-40000") to get a list of available changes and phoebe.update_passband("Bolometric:900-40000") or phoebe.update_all_passbands() to update.


/var/folders/7r/n_x0ntj511v_0gt816mgrc1c0000gq/T/ipykernel_27759/1918838101.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Distance scaling

Observed flux scales as $1/d^2$.

In [4]:
flux_10pc = simulate_observed_flux(bb.intensity, star, np.log10(wavelengths), distance=10.)
flux_100pc = simulate_observed_flux(bb.intensity, star, np.log10(wavelengths), distance=100.)

ratio = float(jnp.median(flux_10pc[:, 0] / flux_100pc[:, 0]))
print(f"flux(10 pc) / flux(100 pc) = {ratio:.1f} (expected 100)")

flux(10 pc) / flux(100 pc) = 100.0 (expected 100)


## Luminosities

Note: these are physically meaningful only when the emulator's `flux`
channel is a true surface flux. `Blackbody.flux` returns the mu=1 intensity,
so the values below serve resolution-convergence comparisons, not absolute
photometry (compare the Stefan-Boltzmann calculation in the Synthetic
Photometry page).

In [5]:
from spice.spectrum import simulate_monochromatic_luminosity, luminosity, absolute_bol_luminosity

lum_wavelengths = np.linspace(200., 40000., 20000)

# L_lambda(lambda): (n_wavelengths, 2), erg/s/Angstrom
mono_lum = simulate_monochromatic_luminosity(bb.flux, star, np.log10(lum_wavelengths))

# Bolometric luminosity in erg/s (trapezoidal integral over wavelength)
L = luminosity(bb.flux, star, lum_wavelengths)

# Absolute bolometric magnitude
M_bol = absolute_bol_luminosity(L)

print(f"L = {float(L):.3e} erg/s, M_bol = {float(M_bol):.2f}")

L = 1.212e+33 erg/s, M_bol = 5.99


## Line-profile variations of non-radial pulsations

Residuals of a rotationally broadened line through one pulsation period: prograde bumps cross the profile blue → red, retrograde red → blue. The figure is written to `docs/img/` in light and dark variants.

In [6]:
from pathlib import Path
from spice.models import IcosphereModel
from spice.models.mesh_transform import add_rotation, add_pulsation, evaluate_pulsations
from spice.spectrum.spectrum import simulate_observed_flux

DOCS_IMG = Path("..") / ".." / "docs" / "img"

def save_doc_fig(name, make_fig):
    """Save make_fig() as <name>.png (light) and <name>_dark.png in docs/img."""
    for style, suffix in [("default", ""), ("dark_background", "_dark")]:
        with plt.style.context(style):
            fig = make_fig()
            fig.savefig(DOCS_IMG / f"{name}{suffix}.png", dpi=120, bbox_inches="tight",
                        facecolor=fig.get_facecolor())
            plt.close(fig)
    print(f"saved {name}.png / {name}_dark.png")

emu = GaussianLineEmulator(line_centers=[5500.], line_widths=[0.05], line_depths=[0.6])
star = IcosphereModel.construct(8000, 2.0, 1.5, emu.to_parameters(), emu.parameter_names)
star = add_rotation(star, 60.)               # equator-on by default

period = 0.2                                 # days
wavelengths = jnp.linspace(5498.6, 5501.4, 600)
velocity = (np.asarray(wavelengths) / 5500. - 1.) * 299792.458
mode = jnp.array([[[0.01, 0.]], [[0.01, 0.]], [[0., 0.]]])   # radial + horizontal
phases = np.linspace(0., 1., 40, endpoint=False)

def residual_spectra(travelling):
    model = add_pulsation(star, 4, 4, period, mode, travelling=travelling)
    profiles = []
    for phase in phases:
        flux = simulate_observed_flux(emu.intensity,
                                      evaluate_pulsations(model, phase * period),
                                      jnp.log10(wavelengths))
        profiles.append(np.asarray(flux[:, 0] / flux[:, 1]))
    profiles = np.array(profiles)
    return profiles - profiles.mean(axis=0)

lpv = {sense: residual_spectra(sense) for sense in ["prograde", "retrograde"]}

[spice] IcosphereModel constructed in 0.4 s
[spice] Pulsations evaluated in 1.8 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.0 s
[spice

In [7]:
def lpv_fig():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    vmax = max(np.abs(r).max() for r in lpv.values())
    for ax, (sense, residuals) in zip(axes, lpv.items()):
        im = ax.imshow(residuals, aspect="auto", origin="lower", cmap="RdBu_r",
                       vmin=-vmax, vmax=vmax,
                       extent=[velocity[0], velocity[-1], phases[0], phases[-1]])
        for v_edge in (-60, 60):
            ax.axvline(v_edge, color="0.5", lw=0.8, ls="--")
        ax.set_title(f"{sense.capitalize()} l = m = 4")
        ax.set_xlabel("Velocity [km/s]")
    axes[0].set_ylabel("Pulsation phase")
    fig.colorbar(im, ax=axes, label="Profile residual")
    return fig

save_doc_fig("travelling_mode_lpv", lpv_fig)

saved travelling_mode_lpv.png / travelling_mode_lpv_dark.png
